# A start at the end — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/python/phenomenon.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/python/phenomenon.html#exercises); this notebook
gives you a running start on each one.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import chaosbook as cb

## The Lorenz system at full parameter freedom

The chapter's figures use the classic parameters $\sigma=10$, $r=28$, $b=8/3$.
Here nothing is precomputed — every slider move integrates the system afresh,
so you can take $\sigma$, $r$ and $b$ anywhere, and shrink the perturbation
$\epsilon = 10^{\,\log_{10}\epsilon}$ of the second initial condition.

In [ ]:
def twin_lorenz(sigma=10.0, r=28.0, b=8 / 3, log10_eps=-3.0, tend=30.0):
    f = lambda t, s: cb.lorenz(t, s, sigma=sigma, r=r, b=b)
    t_eval = np.linspace(0, tend, 3000)
    sol1 = solve_ivp(f, [0, tend], [2, 5, 5], t_eval=t_eval, rtol=1e-9)
    sol2 = solve_ivp(f, [0, tend], [2 + 10.0**log10_eps, 5, 5],
                     t_eval=t_eval, rtol=1e-9)
    plt.figure(figsize=(8, 3))
    plt.plot(sol1.t, sol1.y[0], "b", linewidth=0.8, label="$x(0)=2$")
    plt.plot(sol2.t, sol2.y[0], "r", linewidth=0.8,
             label=f"$x(0)=2+10^{{{log10_eps:.1f}}}$")
    plt.xlabel("$t$")
    plt.ylabel("$x$")
    plt.legend(loc="upper right", frameon=False)
    plt.show()

interact(twin_lorenz,
         sigma=FloatSlider(10.0, min=0.5, max=30.0, step=0.5),
         r=FloatSlider(28.0, min=0.0, max=200.0, step=0.5),
         b=FloatSlider(8 / 3, min=0.1, max=10.0, step=0.1),
         log10_eps=FloatSlider(-3.0, min=-12.0, max=-1.0, step=0.5),
         tend=FloatSlider(30.0, min=5.0, max=100.0, step=5.0));

In [ ]:
def lorenz_attractor(r=28.0, tend=60.0):
    f = lambda t, s: cb.lorenz(t, s, r=r)
    sol = solve_ivp(f, [0, tend], [2, 5, 5],
                    t_eval=np.linspace(0, tend, 6000), rtol=1e-9)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection="3d")
    ax.plot(sol.y[0], sol.y[1], sol.y[2], linewidth=0.4)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_zlabel("$z$")
    plt.show()

# The butterfly only exists for some r: try r below 1, around 20, at 28,
# at 100, at 160...
interact(lorenz_attractor,
         r=FloatSlider(28.0, min=0.5, max=200.0, step=0.5),
         tend=FloatSlider(60.0, min=10.0, max=200.0, step=10.0));

## Exercise: sensitive dependence on initial conditions

Full text: [parts a)–g) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/phenomenon.html#exercises).
The chapter's script is prefilled below; each part needs only a small change.

In [ ]:
# Parts a)-c): the chapter's script. Run it, then change what each part asks:
#   b) plot sol1.y[1] (that is y(t)) or sol1.y[2] (z(t)) instead of sol1.y[0]
#   c) change tstart and tend
tstart, tend = 0, 30
ic1 = [2, 5, 5]
sol1 = solve_ivp(cb.lorenz, [tstart, tend], ic1,
                 t_eval=np.linspace(tstart, tend, 3000), rtol=1e-9)
plt.plot(sol1.t, sol1.y[0])
plt.xlabel("$t$")
plt.ylabel("$x$")
plt.show()

In [ ]:
# Parts d)-e): a second initial condition, epsilon away in x(0).
#   e) reduce epsilon until the two solutions no longer differ
#      (and increase tend above to make sure)
epsilon = 1e-3
ic2 = [ic1[0] + epsilon, ic1[1], ic1[2]]
sol2 = solve_ivp(cb.lorenz, [tstart, tend], ic2,
                 t_eval=np.linspace(tstart, tend, 3000), rtol=1e-9)
plt.plot(sol1.t, sol1.y[0], "b", linewidth=0.8)
plt.plot(sol2.t, sol2.y[0], "r", linewidth=0.8)
plt.xlabel("$t$")
plt.ylabel("$x$")
plt.show()

In [ ]:
# Parts f)-g): the trajectory in phase space.
#   g) change ic1 above to [100, 5, 5] (and try others), re-run the cells,
#      and watch the trajectory converge onto the attractor.
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.plot(sol1.y[0], sol1.y[1], sol1.y[2], linewidth=0.4)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_zlabel("$z$")
plt.show()

## Exercise: the three body problem

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/phenomenon.html#exercises).
As in the chapter, the moon's mass is raised to $m_2 = m_1/4$ to magnify the
chaos (`cb.threebody`'s default); part c) puts the real moon back.

In [ ]:
# The chapter's setup. show_orbit draws the earth (thick black), the moon
# (gray) and the satellite (thin black), optionally in the co-rotating frame.
G, m1, R = 6.67e-11, 5.97e24, 3.84e8
m2 = m1 / 4
two_months = 60 * 24 * 3600

def satellite(x0, tend=two_months, m2=m2, n=8000, u0=0.0, v0=0.0):
    f = lambda t, s: cb.threebody(t, s, G=G, m1=m1, m2=m2, R=R)
    return solve_ivp(f, [0, tend], [x0, 0, u0, v0],
                     t_eval=np.linspace(0, tend, n), rtol=1e-10, atol=1.0)

def show_orbit(sol, m2=m2, rotating=False):
    omega = np.sqrt(G * (m1 + m2) / R**3)
    x1 = -m2 * R / (m1 + m2) * np.cos(omega * sol.t)
    y1 = -m2 * R / (m1 + m2) * np.sin(omega * sol.t)
    x2 = m1 * R / (m1 + m2) * np.cos(omega * sol.t)
    y2 = m1 * R / (m1 + m2) * np.sin(omega * sol.t)
    xs, ys = sol.y[0], sol.y[1]
    if rotating:
        xs, ys = cb.corotating(xs, ys, sol.t, omega)
        x1, y1 = cb.corotating(x1, y1, sol.t, omega)
        x2, y2 = cb.corotating(x2, y2, sol.t, omega)
    plt.plot(x1, y1, "k", linewidth=2.5)
    plt.plot(x2, y2, color="gray", linewidth=1.5)
    plt.plot(xs, ys, "k", linewidth=0.6)
    plt.xlabel("$x$ [m]")
    plt.ylabel("$y$ [m]")
    plt.gca().set_aspect("equal")
    plt.show()

sol = satellite(0.5 * R)
show_orbit(sol)

In [ ]:
# Part a): sweep the initial position (the exercise range is
# x0 in [-5e8, 5e8], i.e. x0/R in about [-1.3, 1.3]) and find the chaotic
# interval. Tick 'rotating' for the co-rotating view (cb.corotating).
def sweep(x0_over_R=0.5, rotating=False):
    show_orbit(satellite(x0_over_R * R), rotating=rotating)

interact(sweep,
         x0_over_R=FloatSlider(0.5, min=-1.3, max=1.3, step=0.02),
         rotating=False);

In [ ]:
# Part b): epsilon = 1 m on x(0) = 2e8 m. How many days until the original
# and perturbed trajectories visibly differ? (sol.t is in seconds)
epsilon = 1.0
sol1 = satellite(2e8)
sol2 = satellite(2e8 + epsilon)
plt.plot(sol1.t / 86400, sol1.y[0], "b", linewidth=0.8)
plt.plot(sol2.t / 86400, sol2.y[0], "r", linewidth=0.8)
plt.xlabel("$t$ [days]")
plt.ylabel("$x$ [m]")
plt.show()

In [ ]:
# Part c): the real moon, m2 = m1/81. Are there still chaotic trajectories?
# Beware: with the light moon a satellite released at rest falls straight
# into the earth (and the integration crawls towards the collision), so
# give it some initial velocity - vary v0 (and x0) and see.
m2_real = 7.36e22
sol = satellite(0.5 * R, m2=m2_real, v0=1000.0)
show_orbit(sol, m2=m2_real)

---
*Back to [the chapter](https://mvreeuwijk.github.io/chaosbook/python/phenomenon.html) — or open the
[cookbook](cookbook.ipynb) in this same lab environment.*